In [1]:
import pandas as pd
import joblib
from datetime import datetime

# load saved model
rf_tuned = joblib.load("../models/rf_tuned.pkl")

# load feature columns from cleaned data so we know exact column structure
df = pd.read_csv("../data/cleaned_adoptions.csv")
X = df.drop(columns=['adopted'])

print("Model and data loaded")
print(f"Features: {X.shape[1]} columns")

Model and data loaded
Features: 96 columns


In [10]:
# build prediction fn

def predict_adoption(
    animal_type,
    is_fixed,
    age=None,
    age_estimate=None,
    breed=None,
    color=None,
    sex=None,
    has_name=None
):
    # age estimation mappings
    AGE_ESTIMATES = {
        'juvenile':        180,   # ~6 months
        'young adult':  730,   # ~2 years
        'mature adult': 1825,  # ~5 years
        'senior':       3650   # ~10 years
    }

    # determine age to use
    if age is not None:
        age_in_days = age
        age_source = f"Established age ({age} days)"
    elif age_estimate is not None:
        age_in_days = AGE_ESTIMATES.get(age_estimate.lower(), 730)
        age_source = f"Estimated age ({age_estimate})"
    else:
        age_in_days = 730
        age_source = "Unknown age (default: young adult)"

    # set defaults for optional fields
    breed = breed or 'Other'
    color = color or 'Black'
    sex = sex or 'Unknown'
    has_name = has_name if has_name is not None else 0

    # automatically use today's date for outcome timing features
    # these are meaningful predictors but unknown at intake
    outcome_month = datetime.now().month
    outcome_weekday = datetime.now().weekday()
    outcome_year = datetime.now().year

    # build input dataframe with all expected columns set to 0
    input_df = pd.DataFrame(0, index=[0], columns=X.columns)

    # fill in numeric features
    input_df['age_in_days'] = age_in_days
    input_df['is_fixed'] = is_fixed
    input_df['has_name'] = has_name
    input_df['outcome_year'] = outcome_year
    input_df['outcome_month'] = outcome_month
    input_df['outcome_weekday'] = outcome_weekday
    input_df['is_mix'] = 1 if 'Mix' in str(breed) else 0
    input_df['is_multicolor'] = 1 if '/' in str(color) else 0

    # set one-hot encoded columns if they exist in our training data
    for col in [f'animal_type_{animal_type}', f'sex_{sex}',
                f'primary_breed_{breed}', f'primary_color_{color.split("/")[0].strip()}']:
        if col in input_df.columns:
            input_df[col] = 1

    # make prediction
    probability = rf_tuned.predict_proba(input_df)[0][1]
    prediction = 'ADOPTED ✅' if probability >= 0.5 else 'NOT ADOPTED ❌'

    # build explanation
    explanations = []

    if is_fixed:
        explanations.append("✅ Animal is fixed: strong positive predictor")
    else:
        explanations.append("⚠️ Animal is not fixed: significantly reduces adoption likelihood")

    if age_in_days <= 180:
        explanations.append(f"✅ {age_source}: young age is the most adoptable range")
    elif age_in_days <= 730:
        explanations.append(f"✅ {age_source}: within a good adoption range")
    elif age_in_days <= 1825:
        explanations.append(f"⚠️ {age_source}: may slightly reduce adoption likelihood")
    else:
        explanations.append(f"⚠️ {age_source}: older age reduces adoption likelihood")

    if has_name:
        explanations.append("✅ Has a name: increases adoption likelihood")
    else:
        explanations.append("⚠️ No name: may reduce adoption likelihood")

    if sex == 'Unknown':
        explanations.append("⚠️ Sex is unknown: reduces adoption likelihood")
    else:
        explanations.append(f"✅ Sex is known: {sex}")

    # print results
    print(f"Prediction: {prediction}")
    print(f"Probability: {probability:.1%} chance of adoption")
    print(f"\nKey factors:")
    for explanation in explanations:
        print(f"  {explanation}")

print("Prediction function ready")

Prediction function ready


In [11]:
# build input fn

def adoption_intake():
    print("🐾 Animal Adoption Predictor\n")
    
    # animal type
    print("Animal type options: Dog, Cat, Bird, Other")
    animal_type = input("Animal type: ").strip().title()
    
    # is fixed
    is_fixed_input = input("Is the animal fixed? (yes/no): ").strip().lower()
    is_fixed = 1 if is_fixed_input == 'yes' else 0
    
    # age
    print("\nAge options:")
    print("  1. Enter established age in days (from vet)")
    print("  2. Use estimated age category")
    age_choice = input("Choose 1 or 2: ").strip()
    
    age = None
    age_estimate = None
    if age_choice == '1':
        age = int(input("Enter age in days: ").strip())
    else:
        print("Estimate options: juvenile, young adult, mature adult, senior")
        age_estimate = input("Age estimate: ").strip().lower()
    
    # breed
    breed = input("\nBreed (press Enter to skip): ").strip() or None
    
    # color
    color = input("Color (press Enter to skip): ").strip() or None
    
    # sex
    print("\nSex options: Male, Female, Unknown")
    sex = input("Sex: ").strip().title() or None
    
    # has name
    has_name_input = input("Does the animal have a name? (yes/no): ").strip().lower()
    has_name = 1 if has_name_input == 'yes' else 0
    
    print("\n" + "="*40)
    predict_adoption(
        animal_type=animal_type,
        is_fixed=is_fixed,
        age=age,
        age_estimate=age_estimate,
        breed=breed,
        color=color,
        sex=sex,
        has_name=has_name
    )

adoption_intake()

🐾 Animal Adoption Predictor

Animal type options: Dog, Cat, Bird, Other


Animal type:  dog
Is the animal fixed? (yes/no):  yes



Age options:
  1. Enter established age in days (from vet)
  2. Use estimated age category


Choose 1 or 2:  2


Estimate options: juvenile, young adult, mature adult, senior


Age estimate:  young adult

Breed (press Enter to skip):  pit bull mix
Color (press Enter to skip):  brindle



Sex options: Male, Female, Unknown


Sex:  female
Does the animal have a name? (yes/no):  yes



Prediction: ADOPTED ✅
Probability: 77.2% chance of adoption

Key factors:
  ✅ Animal is fixed: strong positive predictor
  ✅ Estimated age (young adult): within a good adoption range
  ✅ Has a name: increases adoption likelihood
  ✅ Sex is known: Female


In [9]:
# test fn: young fixed kitten with name

predict_adoption(
    animal_type='Cat',
    is_fixed=1,
    age_estimate='juvenile',
    breed='Domestic Shorthair',
    color='Black',
    sex='Female',
    has_name=1
)

Prediction: ADOPTED ✅
Probability: 89.3% chance of adoption

Key factors:
  ✅ Animal is fixed: strong positive predictor
  ✅ Estimated age (juvenile): young age is the most adoptable range
  ✅ Has a name: increases adoption likelihood
  ✅ Sex is known: Female
